# 03 — Compile the FP32 and TorchAO INT8 CUDA paths

TorchAO's GPU tutorial recommends `torch.compile` for optimized quantized execution, but **it may fail, compile slowly, use significant memory, or remain slower** for BERT/your exact GPU and library versions.

This experiment is intentionally separate from the baseline. It does NOT overwrite eager benchmark results and does NOT silently fall back to eager when compilation fails.

In [1]:
from pathlib import Path
import subprocess, sys, json

def find_analysis_folder():
    current = Path.cwd().resolve()
    for parent in [current, *current.parents]:
        for relative in [Path('notebooks/runtime_analysis'), Path('runtime_analysis'), Path('.')]:
            candidate = (parent / relative).resolve()
            if (candidate / 'config.json').is_file() and (candidate / 'worker.py').is_file():
                return candidate
    raise FileNotFoundError('Cannot locate notebooks/runtime_analysis/config.json. Open this notebook inside the copied repository.')

ANALYSIS_DIR = find_analysis_folder()
REPO_ROOT = ANALYSIS_DIR.parent.parent
CONFIG = json.loads((ANALYSIS_DIR/'config.json').read_text())
print('Folder:', ANALYSIS_DIR)
print('Repository:', REPO_ROOT)

Folder: /home/mohnishr/workspace/DLBA/Quantization-Spurious-Heuristics-in-Transformers/notebooks/runtime_analysis
Repository: /home/mohnishr/workspace/DLBA/Quantization-Spurious-Heuristics-in-Transformers


## 🔴 Run compiled experiments for seed 42 first

Compare `fp32_compiled` against `int8_compiled` only. Both share the same compiler settings in `config.json` (default `max-autotune`, `dynamic=true`). Compilation/warm-up cost is recorded separately from steady-state inference.

In [2]:
import subprocess
import sys

SEEDS = [42]

cmd = [
    sys.executable,
    str(ANALYSIS_DIR / "run_experiments.py"),
    "--seeds", *map(str, SEEDS),
    "--variants",
    "fp32_compiled",
    "int8_compiled_fp32_head",
    "--task", "benchmark",
]

subprocess.run(cmd, cwd=REPO_ROOT, check=True)

CUDA variants run SEQUENTIALLY in a FRESH PROCESS each.
Working directory: /home/mohnishr/workspace/DLBA/Quantization-Spurious-Heuristics-in-Transformers

Running seed=42, variant=fp32_compiled, task=benchmark



W1025 23:05:59.480000 48077 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1025 23:05:59.497000 48077 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 18145.44it/s]


  fp32_compiled repeat 1: 30000 examples, 11.405s, 2630.5/s
  fp32_compiled repeat 2: 30000 examples, 9.351s, 3208.3/s
  fp32_compiled repeat 3: 30000 examples, 9.366s, 3202.9/s
Saved: /home/mohnishr/workspace/DLBA/Quantization-Spurious-Heuristics-in-Transformers/results/runtime_analysis/seed42/fp32_compiled/benchmark_summary.json
Finished exit code 0 in 44.3s

Running seed=42, variant=int8_compiled_fp32_head, task=benchmark



W1025 23:06:43.744000 48304 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1025 23:06:43.761000 48304 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 15129.12it/s]
/home/mohnishr/workspace/DLBA/Quantization-Spurious-Heuristics-in-Transformers/.venv/lib/python3.12/site-packages/torch/_functorch/_aot_autograd/autograd_cache.py:707: UserWarning: Int8Tensor does not implement _stable_hash_for_caching. For PT2-compatible tensor subclasses, it is recommended to implement _stable_hash_for_caching(self) -> str for stable AOT autograd cach

  int8_compiled_fp32_head repeat 1: 30000 examples, 7.745s, 3873.6/s
  int8_compiled_fp32_head repeat 2: 30000 examples, 2.225s, 13482.3/s
  int8_compiled_fp32_head repeat 3: 30000 examples, 2.223s, 13492.3/s
Saved: /home/mohnishr/workspace/DLBA/Quantization-Spurious-Heuristics-in-Transformers/results/runtime_analysis/seed42/int8_compiled_fp32_head/benchmark_summary.json
Finished exit code 0 in 43.2s

All requested subprocess jobs completed successfully.


CompletedProcess(args=['/home/mohnishr/workspace/DLBA/Quantization-Spurious-Heuristics-in-Transformers/.venv/bin/python', '/home/mohnishr/workspace/DLBA/Quantization-Spurious-Heuristics-in-Transformers/notebooks/runtime_analysis/run_experiments.py', '--seeds', '42', '--variants', 'fp32_compiled', 'int8_compiled_fp32_head', '--task', 'benchmark'], returncode=0)

## Refresh matched-mode comparisons

If either worker fails, open its `benchmark_status.json`, preserve the failure as a finding, and **do not compare compiled INT8 to eager FP32 to claim a quantization benefit**.

In [3]:
subprocess.run([sys.executable, str(ANALYSIS_DIR/'summarize_results.py')], cwd=REPO_ROOT, check=True)
import pandas as pd
comparison = REPO_ROOT / CONFIG['results_dir'] / 'paired_fp32_int8_comparison.csv'
display(pd.read_csv(comparison))

Matched comparisons:
  seed                    mode  fp32_variant            int8_variant  int8_over_fp32_throughput_x
   42     eager_original_int8    fp32_eager              int8_eager                     0.187956
   42 compiled_fp32_head_int8 fp32_compiled int8_compiled_fp32_head                     4.209367
Saved: /home/mohnishr/workspace/DLBA/Quantization-Spurious-Heuristics-in-Transformers/results/runtime_analysis/all_benchmark_variants.csv
Saved: /home/mohnishr/workspace/DLBA/Quantization-Spurious-Heuristics-in-Transformers/results/runtime_analysis/benchmark_inclusion_status.csv
Saved: /home/mohnishr/workspace/DLBA/Quantization-Spurious-Heuristics-in-Transformers/results/runtime_analysis/pair_eligibility_status.csv
NOTE: FP32-head INT8 is a DISTINCT quantization variant; check 04 prediction equivalence before relating to original HANS behavioral results.


,seed,mode,fp32_variant,int8_variant,comparison_note,throughput_batch_size,num_hans_examples,partial_batch_policy,compile_mode,device,...,int8_batch32_median_forward_latency_ms,int8_change_pct_batch32_median_forward_latency_ms,fp32_batch32_p95_forward_latency_ms,int8_batch32_p95_forward_latency_ms,int8_change_pct_batch32_p95_forward_latency_ms,fp32_runtime_quantization_setup_seconds,int8_runtime_quantization_setup_seconds,int8_change_pct_runtime_quantization_setup_seconds,int8_over_fp32_throughput_x,int8_inference_speedup_x
0,42,eager_original_int8,fp32_eager,int8_eager,Original W8A8 dynamic INT8 versus FP32; same e...,32,30000,repeat examples within incomplete final batch ...,NaN,cuda,...,58.711331,471.241007,12.827604,59.889633,366.880910,0.0,0.180217,NaN,0.187956,0.187956
1,42,compiled_fp32_head_int8,fp32_compiled,int8_compiled_fp32_head,Modified W8A8 (FP32 classifier) versus FP32; s...,32,30000,repeat examples within incomplete final batch ...,max-autotune-no-cudagraphs,cuda,...,1.985370,-80.036013,12.328203,2.652528,-78.484065,0.0,0.893113,NaN,4.209367,4.209367


## Next

Run **`04_prediction_validation.ipynb`**. Any execution-path change that alters HANS predictions needs to be measured before linking runtime metrics with your prior behavioral study.